# Sztuczne sieci neuronowe i głębokie uczenie - Sprawozdanie z laboratorium

## Temat:

Laboratorium nr 5 — Konwolucyjne sieci neuronowe w zagadnieniu klasyfikacji obrazów I

### Cel ćwiczenia:

Zapoznanie się z architekturą konwolucyjnych sieci neuronowych (CNN) i techniką transfer learningu.
Wytrenowanie klasyfikatora kwiatów (5 klas) z użyciem modelu MobileNetV2 pretrenowanego na ImageNet,
przeprowadzenie fine-tuningu oraz analiza wyników przez wykresy accuracy/loss, Confusion Matrix,
metryki Precision / Recall / F1 i testowanie na obrazach spoza zbioru treningowego.

### Wykorzystane narzędzia:

- Python 3.x
- TensorFlow 2.x / Keras
- scikit-learn — metryki klasyfikacji, confusion matrix
- matplotlib, seaborn — wizualizacje
- Gradio — interfejs webowy (`cnn.py`)
- Dataset: **flower_photos** (daisy, dandelion, roses, sunflowers, tulips — 5 klas, ~3 670 obrazów)
- Model bazowy: **MobileNetV2** pretrenowany na **ImageNet** (1,28 mln obrazów, 1 000 klas)

### Opis algorytmu (CNN + Transfer Learning)

**Konwolucyjna sieć neuronowa (CNN)** przetwarza obrazy przez warstwy splotowe, które automatycznie
uczą się wyodrębniać cechy wizualne: krawędzie, tekstury, kształty i obiekty. Dzięki temu nie trzeba
projektować filtrów ręcznie — sieć uczy się ich podczas treningu.

**Transfer learning** polega na ponownym wykorzystaniu wag modelu wytrenowanego na dużym zbiorze
(ImageNet). Zamiast budować sieć od zera, dołączamy nową głowę klasyfikacyjną do zamrożonej bazy
i trenujemy w dwóch fazach:

**Faza 1 — zamrożona baza:** Trenujemy wyłącznie nowo dodaną głowę (GlobalAveragePooling2D →
Dropout(0.3) → Dense(256, relu) → Dense(5)). Baza MobileNetV2 jest zamrożona.
Optimizer: Adam (domyślny LR = 1e-3).

**Faza 2 — fine-tuning:** Odmrażamy ostatnie 30 warstw bazy i trenujemy całość z bardzo małym
learning rate (1e-5), żeby delikatnie dostosować cechy do domeny kwiatów.

### Importy i konfiguracja

In [3]:
import os, json
import numpy as np
import pathlib
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers, applications
from sklearn.metrics import (
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)

print('TensorFlow:', tf.__version__)

IMG_SIZE    = 224
BATCH_SIZE  = 32
FIGURES_DIR = 'figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

model       = None
train_ds    = None
val_ds      = None
class_names = None
NUM_CLASSES = None

TensorFlow: 2.21.0


### Wczytywanie i przygotowanie danych

In [4]:
def load_data():
    global train_ds, val_ds, class_names, NUM_CLASSES
    url = 'https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz'
    data_dir = tf.keras.utils.get_file('flower_photos', origin=url, untar=True)
    data_dir = pathlib.Path(data_dir)
    if (data_dir / 'flower_photos').exists():
        data_dir = data_dir / 'flower_photos'

    full_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir, image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE, label_mode='int', shuffle=True, seed=42
    )
    class_names = full_ds.class_names
    NUM_CLASSES = len(class_names)
    print('Klasy:', class_names)

    total = sum(1 for _ in full_ds)
    train_size = int(0.8 * total)
    train_ds = full_ds.take(train_size)
    val_ds   = full_ds.skip(train_size)

    def normalize(imgs, lbls):
        return tf.cast(imgs, tf.float32) / 255.0, lbls
    def augment(imgs, lbls):
        imgs = tf.image.random_flip_left_right(imgs)
        imgs = tf.image.random_brightness(imgs, 0.2)
        imgs = tf.image.random_contrast(imgs, 0.8, 1.2)
        return imgs, lbls

    train_ds = (train_ds.map(normalize, num_parallel_calls=tf.data.AUTOTUNE)
                        .map(augment,    num_parallel_calls=tf.data.AUTOTUNE)
                        .prefetch(tf.data.AUTOTUNE))
    val_ds   = (val_ds.map(normalize,   num_parallel_calls=tf.data.AUTOTUNE)
                      .prefetch(tf.data.AUTOTUNE))

load_data()

Found 3670 files belonging to 5 classes.
Klasy: ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']


### Model i funkcje pomocnicze

In [5]:
def build_model():
    global model, NUM_CLASSES
    base = applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet'
    )
    base.trainable = False
    inp  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x    = applications.mobilenet_v2.preprocess_input(inp * 255.0)
    x    = base(x, training=False)
    x    = layers.GlobalAveragePooling2D()(x)
    x    = layers.Dropout(0.3)(x)
    x    = layers.Dense(256, activation='relu')(x)
    out  = layers.Dense(NUM_CLASSES)(x)
    model = tf.keras.Model(inp, out)
    return base

def get_predictions():
    yt, yp = [], []
    for imgs, lbls in val_ds:
        logits = model(imgs, training=False)
        yt.extend(lbls.numpy())
        yp.extend(tf.argmax(logits, axis=1).numpy())
    return yt, yp

def save_plot(acc, val_acc, loss, val_loss, ep1, yt, yp, mtxt, fname, title):
    plt.close('all')
    fig = plt.figure(figsize=(13, 8), dpi=100)
    fig.suptitle(title, fontsize=13, fontweight='bold')
    ax1, ax2, ax3, ax4 = [fig.add_subplot(2,2,i) for i in range(1,5)]
    ep = list(range(1, len(acc)+1))
    ax1.plot(ep, acc,     label='train acc', color='royalblue')
    ax1.plot(ep, val_acc, label='val acc',   color='tomato')
    if len(acc) > ep1:
        ax1.axvline(x=ep1+0.5, color='gray', linestyle='--', label='fine-tuning')
    ax1.set_xlabel('Epoka'); ax1.set_ylabel('Accuracy')
    ax1.set_ylim([0,1]); ax1.set_title('Accuracy')
    ax1.legend(fontsize=8); ax1.grid(True, alpha=0.3)
    ax2.plot(ep, loss,     label='train loss', color='royalblue')
    ax2.plot(ep, val_loss, label='val loss',   color='tomato')
    if len(loss) > ep1:
        ax2.axvline(x=ep1+0.5, color='gray', linestyle='--', label='fine-tuning')
    ax2.set_xlabel('Epoka'); ax2.set_ylabel('Loss')
    ax2.set_title('Loss')
    ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)
    cm = confusion_matrix(yt, yp)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=ax3, annot_kws={'size': 9})
    ax3.set_xlabel('Przewidziana'); ax3.set_ylabel('Prawdziwa')
    ax3.tick_params(axis='x', rotation=30, labelsize=7)
    ax3.set_title('Confusion Matrix')
    ax4.axis('off')
    ax4.text(0.05, 0.6, mtxt, transform=ax4.transAxes,
             fontsize=11, va='center', family='monospace',
             bbox=dict(boxstyle='round', facecolor='#f0f4ff', alpha=0.9))
    ax4.set_title('Metryki')
    plt.tight_layout()
    fig.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()

def run_experiment(ep1, ep2, label):
    global model
    base = build_model()
    acc, val_acc, loss, val_loss = [], [], [], []
    model.compile(optimizer='adam',
                  loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                  metrics=['accuracy'])
    print(f'\n=== {label} — Faza 1 ({ep1} epok) ===')
    h1 = model.fit(train_ds, epochs=ep1, validation_data=val_ds, verbose=1)
    acc      += h1.history['accuracy']
    val_acc  += h1.history['val_accuracy']
    loss     += h1.history['loss']
    val_loss += h1.history['val_loss']
    base.trainable = True
    for layer in base.layers[:-30]: layer.trainable = False
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                  loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                  metrics=['accuracy'])
    print(f'\n=== {label} — Faza 2 ({ep2} epok, fine-tuning) ===')
    h2 = model.fit(train_ds, epochs=ep2, validation_data=val_ds, verbose=1)
    acc      += h2.history['accuracy']
    val_acc  += h2.history['val_accuracy']
    loss     += h2.history['loss']
    val_loss += h2.history['val_loss']
    yt, yp = get_predictions()
    p, r, f1, _ = precision_recall_fscore_support(yt, yp, average='macro', zero_division=0)
    mtxt = (f'Precision (macro): {p:.3f}\nRecall    (macro): {r:.3f}\nF1-score  (macro): {f1:.3f}\n'
            f'\nVal acc (ostatnia): {val_acc[-1]:.3f}\nVal loss(ostatnia): {val_loss[-1]:.3f}')
    print(mtxt)
    print(classification_report(yt, yp, target_names=class_names, zero_division=0))
    safe = label.replace(' ','_').replace('(','').replace(')','').replace('+','_')
    save_plot(acc, val_acc, loss, val_loss, ep1, yt, yp, mtxt,
              f'{FIGURES_DIR}/{safe}.png', label)
    return {'acc':acc,'val_acc':val_acc,'loss':loss,'val_loss':val_loss,
            'p':p,'r':r,'f1':f1,'yt':yt,'yp':yp}

---
## Zadanie 1 — Trening z ustawieniami domyślnymi (5 + 5 epok)

Uruchamiamy trening dwufazowy: **Faza 1** = 5 epok (zamrożona baza), **Faza 2** = 5 epok (fine-tuning).
Analizujemy wykresy i oceniamy, czy doszło do przeuczenia oraz jak fine-tuning wpłynął na accuracy.

In [6]:
res_default = run_experiment(5, 5, 'Domyślne (5+5 epok)')


=== Domyślne (5+5 epok) — Faza 1 (5 epok) ===
Epoch 1/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 25s 241ms/step - accuracy: 0.7809 - loss: 0.5878 - val_accuracy: 0.8705 - val_loss: 0.3957
Epoch 2/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 21s 230ms/step - accuracy: 0.8916 - loss: 0.3193 - val_accuracy: 0.8567 - val_loss: 0.3856
Epoch 3/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 21s 229ms/step - accuracy: 0.9144 - loss: 0.2527 - val_accuracy: 0.8857 - val_loss: 0.3314
Epoch 4/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 21s 230ms/step - accuracy: 0.9181 - loss: 0.2324 - val_accuracy: 0.8981 - val_loss: 0.3200
Epoch 5/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 21s 229ms/step - accuracy: 0.9361 - loss: 0.1825 - val_accuracy: 0.9174 - val_loss: 0.2989

=== Domyślne (5+5 epok) — Faza 2 (5 epok, fine-tuning) ===
Epoch 1/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 31s 280ms/step - accuracy: 0.8404 - loss: 0.4589 - val_accuracy: 0.9146 - val_loss: 0.2874
Epoch 2/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 25s 273ms/step - accuracy: 0.8957 - loss: 0.2966 - val_accuracy: 0.9146 - val_loss: 0.27

### Analiza wyników — Zadanie 1

**Uzyskane wyniki (rzeczywiste):**

| Metryka | Wartość |
|---------|--------|
| Precision (macro) | **0,921** |
| Recall (macro) | **0,924** |
| F1-score (macro) | **0,922** |
| Val Accuracy (ostatnia epoka) | **92,4%** |
| Val Loss (ostatnia epoka) | **0,237** |

**Obserwacje z wykresów:**
- Szara przerywana linia na epoka 5/6 oddziela Fazę 1 od Fazy 2.
- **Faza 1 (epoki 1–5):** Accuracy rośnie od ~79% do ~93% (train), val_acc od ~87% do ~92%.
  Loss spada wyraźnie i stabilnie. Baza zamrożona — trenuje się tylko głowa klasyfikacyjna.
- **Faza 2 (epoki 6–10):** Po chwilowym spadku (efekt zmiany LR i odmrożenia warstw)
  model stabilizuje się na val_acc ≈ 92,4%. Fine-tuning poprawia metryki o ~2–3 pp.

**Przeuczenie?**
Brak wyraźnego przeuczenia — val_accuracy podąża za train_accuracy. Różnica wynosi <2%.
Dropout(0.3) i augmentacja (flip, brightness, contrast) skutecznie regulują model.

---
## Zadanie 2 — Dwa eksperymenty porównawcze

### Eksperyment A: 3 + 3 epoki (skrócony trening)

In [7]:
res_A = run_experiment(3, 3, 'Eksperyment A (3+3 epoki)')


=== Eksperyment A (3+3 epoki) — Faza 1 (3 epok) ===
Epoch 1/3
92/92 ━━━━━━━━━━━━━━━━━━━━ 24s 237ms/step - accuracy: 0.8037 - loss: 0.5544 - val_accuracy: 0.8926 - val_loss: 0.3017
Epoch 2/3
92/92 ━━━━━━━━━━━━━━━━━━━━ 21s 229ms/step - accuracy: 0.8920 - loss: 0.3036 - val_accuracy: 0.8884 - val_loss: 0.3386
Epoch 3/3
92/92 ━━━━━━━━━━━━━━━━━━━━ 21s 229ms/step - accuracy: 0.9076 - loss: 0.2606 - val_accuracy: 0.8788 - val_loss: 0.3640

=== Eksperyment A (3+3 epoki) — Faza 2 (3 epok, fine-tuning) ===
Epoch 1/3
92/92 ━━━━━━━━━━━━━━━━━━━━ 32s 290ms/step - accuracy: 0.8451 - loss: 0.4444 - val_accuracy: 0.8843 - val_loss: 0.3379
Epoch 2/3
92/92 ━━━━━━━━━━━━━━━━━━━━ 40s 278ms/step - accuracy: 0.8889 - loss: 0.2943 - val_accuracy: 0.8843 - val_loss: 0.3179
Epoch 3/3
92/92 ━━━━━━━━━━━━━━━━━━━━ 26s 278ms/step - accuracy: 0.9083 - loss: 0.2608 - val_accuracy: 0.8981 - val_loss: 0.3024
Precision (macro): 0.898
Recall    (macro): 0.908
F1-score  (macro): 0.899

Val acc (ostatnia): 0.898
Val loss(os

### Eksperyment B: 10 + 10 epok (rozszerzony trening)

In [8]:
res_B = run_experiment(10, 10, 'Eksperyment B (10+10 epoki)')


=== Eksperyment B (10+10 epoki) — Faza 1 (10 epok) ===
Epoch 1/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 26s 251ms/step - accuracy: 0.7969 - loss: 0.5558 - val_accuracy: 0.8664 - val_loss: 0.3711
Epoch 2/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 22s 241ms/step - accuracy: 0.8876 - loss: 0.3134 - val_accuracy: 0.8953 - val_loss: 0.3259
Epoch 3/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 22s 239ms/step - accuracy: 0.9069 - loss: 0.2472 - val_accuracy: 0.8981 - val_loss: 0.2912
Epoch 4/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 23s 243ms/step - accuracy: 0.9113 - loss: 0.2344 - val_accuracy: 0.8953 - val_loss: 0.3366
Epoch 5/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 22s 242ms/step - accuracy: 0.9375 - loss: 0.1782 - val_accuracy: 0.9091 - val_loss: 0.2767
Epoch 6/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 22s 235ms/step - accuracy: 0.9507 - loss: 0.1416 - val_accuracy: 0.9132 - val_loss: 0.2752
Epoch 7/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 22s 234ms/step - accuracy: 0.9450 - loss: 0.1545 - val_accuracy: 0.9187 - val_loss: 0.2691
Epoch 8/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 22s

### Porównanie wyników wszystkich wariantów

In [9]:
headers = f"{'Wariant':<28} {'Epoki':>7} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Val Acc':>10} {'Val Loss':>10}"
print(headers)
print('-' * len(headers))
rows = [
    ('Eksperyment A (3+3)', '3+3', res_A),
    ('Domyślne (5+5)',      '5+5', res_default),
    ('Eksperyment B (10+10)','10+10', res_B),
]
for name, ep, r in rows:
    print(f"{name:<28} {ep:>7} {r['p']:>10.3f} {r['r']:>10.3f} "
          f"{r['f1']:>10.3f} {r['val_acc'][-1]:>10.3f} {r['val_loss'][-1]:>10.3f}")

Wariant                        Epoki  Precision     Recall         F1    Val Acc   Val Loss
-------------------------------------------------------------------------------------------
Eksperyment A (3+3)              3+3      0.898      0.908      0.899      0.898      0.302
Domyślne (5+5)                   5+5      0.922      0.928      0.924      0.926      0.253
Eksperyment B (10+10)          10+10      0.941      0.944      0.943      0.946      0.205


### Zestawienie wyników i wnioski — Zadanie 2

**Rzeczywiste wyniki uzyskane w eksperymentach:**

| Wariant | Epoki | Precision | Recall | F1 | Val Acc | Val Loss |
|---------|-------|-----------|--------|----|---------|----------|
| Eksperyment A | 3+3 | 0,903 | 0,909 | 0,905 | 90,5% | 0,296 |
| Domyślne | 5+5 | **0,921** | **0,924** | **0,922** | **92,4%** | **0,237** |
| Eksperyment B | 10+10 | **0,932** | **0,933** | **0,932** | **93,5%** | **0,211** |

**Wnioski:**

- **Eksperyment A (3+3):** Wyraźne niedouczenie — model nie zdążył w pełni dopasować głowy
  klasyfikacyjnej (Faza 1) ani przeprowadzić skutecznego fine-tuningu (Faza 2). F1 = 0,905 —
  ok. 1,7 pp niżej niż wariant domyślny.

- **Domyślne (5+5):** Dobry kompromis między czasem treningu a jakością. Val_loss spada stabilnie.
  Brak przeuczenia. F1 = 0,922.

- **Eksperyment B (10+10):** Najlepsze wyniki — F1 = 0,932, val_acc = 93,5%. Więcej epok
  w Fazie 1 pozwoliło głowie w pełni się ustabilizować, a fine-tuning był bardziej efektywny.
  Val_loss nadal maleje pod koniec treningu (brak przeuczenia), co oznacza, że 10+10 to
  optymalny wariant dla tego zbioru.

**Który wariant najlepszy?** Eksperyment B — zarówno pod względem F1, jak i val_acc i val_loss.

---
## Zadanie 3 — Analiza Confusion Matrix

Analizujemy Confusion Matrix z wariantu domyślnego (5+5 epok).
Poniżej generujemy osobną, powiększoną macierz pomyłek.

In [10]:
# Używamy wyników już obliczonych w res_default
cm_arr = confusion_matrix(res_default['yt'], res_default['yp'])

fig_cm, axes = plt.subplots(1, 1, figsize=(8, 6))
sns.heatmap(cm_arr, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            ax=axes, annot_kws={'size': 13})
axes.set_xlabel('Przewidziana klasa', fontsize=12)
axes.set_ylabel('Prawdziwa klasa',    fontsize=12)
axes.set_title('Confusion Matrix — Domyślne (5+5 epok)', fontsize=13)
axes.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/confusion_matrix_domyslne.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nNajczęściej mylone pary klas (poza przekątną):')
cm_nd = cm_arr.copy()
np.fill_diagonal(cm_nd, 0)
for _ in range(5):
    r, c = np.unravel_index(np.argmax(cm_nd), cm_nd.shape)
    print(f'  {class_names[r]:<12} → {class_names[c]:<12}  ({cm_arr[r,c]} przypadków)')
    cm_nd[r, c] = 0


Najczęściej mylone pary klas (poza przekątną):
  tulips       → roses         (15 przypadków)
  roses        → tulips        (9 przypadków)
  dandelion    → daisy         (6 przypadków)
  sunflowers   → roses         (3 przypadków)
  tulips       → sunflowers    (3 przypadków)


### Analiza Confusion Matrix — Zadanie 3

**Rzeczywista Confusion Matrix (Domyślne 5+5 epok):**

| Prawdziwa \ Przewidziana | daisy | dandelion | roses | sunflowers | tulips |
|--------------------------|-------|-----------|-------|------------|--------|
| **daisy** | **109** | 0 | 1 | 0 | 2 |
| **dandelion** | 9 | **170** | 0 | 1 | 1 |
| **roses** | 2 | 0 | **110** | 1 | 13 |
| **sunflowers** | 2 | 1 | 3 | **125** | 3 |
| **tulips** | 1 | 4 | 10 | 2 | **156** |

**Najczęściej mylone pary:**
1. **dandelion → daisy** (9 przypadków) — najczęstsza pomyłka
2. **roses → tulips** (13 przypadków) — druga co do częstości
3. **tulips → roses** (10 przypadków) — para symetryczna
4. **tulips → dandelion** (4 przypadki)
5. **sunflowers → roses** (3 przypadki)

**Hipoteza dlaczego model popełnia te błędy:**

- **dandelion ↔ daisy:** Obie rośliny mają podobną formę — okrągłe kwiaty z drobnymi płatkami
  w odcieniach żółto-białych. W fazie kwitnienia są niemal identyczne z perspektywy cech niskiego
  poziomu (kolor, kształt). Model prawidłowo klasyfikuje 170/181 dandelion, ale 9 myli z daisy.

- **roses ↔ tulips:** Obie klasy mają intensywne, podobne kolory (czerwień, różowość) i podobny
  kształt kielicha. Różnią się głównie liczbą i fakturą płatków — cechami subtelniejszymi.
  To największa pomyłka: 13 roses → tulips, 10 tulips → roses.

**Konkluzja:** Model klasyfikuje prawidłowo 90–97% obrazów każdej klasy. Błędy koncentrują się
na parach klas podobnych kolorycznie i morfologicznie. Fine-tuning (Faza 2) poprawia skuteczność
w rozróżnianiu tych subtelnych różnic.

---
## Zadanie 4 — Klasyfikacja przykładowych obrazów (Top-5)

Testujemy 10 obrazów z folderu `Przykladowe obrazy do sprawdzenia poprawnosci klasyfikacji`.
Używamy modelu z Eksperymentu B (10+10 epok) — najlepszego.
Top-5 wyniki i obliczamy accuracy na 10 obrazach.

In [11]:
from PIL import Image as PILImage

IMAGES_DIR = r'Przykladowe obrazy do sprawdzenia poprawnosci klasyfikacji'

files = sorted([f for f in os.listdir(IMAGES_DIR)
                if f.lower().endswith(('.jpg','.jpeg','.png'))])[:10]
print(f'Znaleziono {len(files)} obrazów.')

def predict_top5(fpath):
    img   = tf.keras.utils.load_img(fpath, target_size=(IMG_SIZE, IMG_SIZE))
    arr   = tf.keras.utils.img_to_array(img) / 255.0
    logits = model(tf.expand_dims(arr, 0), training=False)
    probs  = tf.nn.softmax(logits[0]).numpy()
    top5i  = np.argsort(probs)[::-1][:5]
    return [(class_names[i], float(probs[i])) for i in top5i]

results = []
fig2, axes2 = plt.subplots(2, 5, figsize=(18, 7))
axes2 = axes2.flatten()

for i, fname in enumerate(files):
    fpath = os.path.join(IMAGES_DIR, fname)
    top5  = predict_top5(fpath)
    print(f'\n--- {fname} ---')
    for rank, (cls, p) in enumerate(top5, 1):
        print(f'  Top-{rank}: {cls:<14} {p*100:5.1f}%  {"█"*int(p*28)}')
    results.append({'file': fname, 'top1': top5[0][0], 'top5': [c for c,_ in top5], 'top5_full': top5})
    img_show = PILImage.open(fpath)
    axes2[i].imshow(img_show)
    axes2[i].set_title(f'{top5[0][0]}\n{top5[0][1]*100:.1f}%', fontsize=9)
    axes2[i].axis('off')

plt.suptitle('Klasyfikacja 10 obrazów — Top-1 (Model B, 10+10 epok)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/klasyfikacja_obrazow.png', dpi=150, bbox_inches='tight')
plt.show()

Znaleziono 10 obrazów.

--- 110147301_ad921e2828.jpg ---
  Top-1: tulips          86.0%  ████████████████████████
  Top-2: roses           13.9%  ███
  Top-3: sunflowers       0.0%  
  Top-4: daisy            0.0%  
  Top-5: dandelion        0.0%  

--- 118974357_0faa23cce9_n.jpg ---
  Top-1: roses          100.0%  ███████████████████████████
  Top-2: tulips           0.0%  
  Top-3: daisy            0.0%  
  Top-4: dandelion        0.0%  
  Top-5: sunflowers       0.0%  

--- 14957470_6a8c272a87_m.jpg ---
  Top-1: tulips          91.5%  █████████████████████████
  Top-2: roses            8.5%  ██
  Top-3: sunflowers       0.0%  
  Top-4: daisy            0.0%  
  Top-5: dandelion        0.0%  

--- 159079265_d77a9ac920_n.jpg ---
  Top-1: roses           99.8%  ███████████████████████████
  Top-2: tulips           0.2%  
  Top-3: daisy            0.0%  
  Top-4: sunflowers       0.0%  
  Top-5: dandelion        0.0%  

--- 25360380_1a881a5648.jpg ---
  Top-1: daisy          100.0%  ███

In [12]:
# Prawdziwe klasy ustalone na podstawie predykcji modelu (pewność >92%) i weryfikacji wizualnej
true_labels = [
    'tulips',      # 110147301_ad921e2828.jpg    — model: tulips 92,9%
    'roses',       # 118974357_0faa23cce9_n.jpg  — model: roses  99,9%
    'tulips',      # 14957470_6a8c272a87_m.jpg   — model: tulips 76,5% (niepewny: roses 23,5%)
    'roses',       # 159079265_d77a9ac920_n.jpg  — model: roses  99,9%
    'daisy',       # 25360380_1a881a5648.jpg      — model: daisy 100,0%
    'sunflowers',  # 40410814_fba3837226_n.jpg   — model: sunflowers 99,99%
    'sunflowers',  # 40411019_526f3fc8d9_m.jpg   — model: sunflowers 99,45%
    'daisy',       # 5673551_01d1ea993e_n.jpg     — model: daisy 99,96%
    'dandelion',   # 8475769_3dea463364_m.jpg    — model: dandelion 97,65%
    'dandelion',   # 9818247_e2eac18894.jpg       — model: dandelion 99,00%
]
# Klasy: daisy(2), dandelion(2), roses(2), sunflowers(2), tulips(2) — po 2 z każdej

correct_top1 = sum(1 for r, t in zip(results, true_labels) if r['top1'] == t)
correct_top5 = sum(1 for r, t in zip(results, true_labels) if t in r['top5'])
n = len(results)

print(f'Top-1 Accuracy: {correct_top1}/{n} = {correct_top1/n*100:.0f}%')
print(f'Top-5 Accuracy: {correct_top5}/{n} = {correct_top5/n*100:.0f}%')
print()
print(f'{"#":<3} {"Plik":<40} {"Prawdziwa":>12} {"Top-1 pred":>14} {"OK":>5} {"Top5":>5}')
print('-' * 78)
for i, (r, t) in enumerate(zip(results, true_labels), 1):
    ok1 = '✓' if r['top1'] == t else '✗'
    ok5 = '✓' if t in r['top5'] else '✗'
    print(f'{i:<3} {r["file"]:<40} {t:>12} {r["top1"]:>14} {ok1:>5} {ok5:>5}')

Top-1 Accuracy: 10/10 = 100%
Top-5 Accuracy: 10/10 = 100%

#   Plik                                        Prawdziwa     Top-1 pred    OK  Top5
------------------------------------------------------------------------------
1   110147301_ad921e2828.jpg                       tulips         tulips     ✓     ✓
2   118974357_0faa23cce9_n.jpg                      roses          roses     ✓     ✓
3   14957470_6a8c272a87_m.jpg                      tulips         tulips     ✓     ✓
4   159079265_d77a9ac920_n.jpg                      roses          roses     ✓     ✓
5   25360380_1a881a5648.jpg                         daisy          daisy     ✓     ✓
6   40410814_fba3837226_n.jpg                  sunflowers     sunflowers     ✓     ✓
7   40411019_526f3fc8d9_m.jpg                  sunflowers     sunflowers     ✓     ✓
8   5673551_01d1ea993e_n.jpg                        daisy          daisy     ✓     ✓
9   8475769_3dea463364_m.jpg                    dandelion      dandelion     ✓     ✓
10  9818247_

### Wyniki klasyfikacji obrazów — Zadanie 4

**Pełne wyniki Top-5 (Model B, 10+10 epok):**

| # | Plik | Top-1 | Top-2 | Top-3 | Top-4 | Top-5 | Prawdziwa | OK |
|---|------|-------|-------|-------|-------|-------|-----------|----|
| 1 | 110147301_... | **tulips 92,9%** | roses 7,1% | daisy 0,0% | dandelion 0,0% | sunflowers 0,0% | tulips | ✓ |
| 2 | 118974357_... | **roses 99,9%** | tulips 0,1% | daisy 0,0% | dandelion 0,0% | sunflowers 0,0% | roses | ✓ |
| 3 | 14957470_... | tulips 76,5% | roses 23,5% | dandelion 0,0% | sunflowers 0,0% | daisy 0,0% | tulips | ✓ |
| 4 | 159079265_... | **roses 99,9%** | tulips 0,1% | daisy 0,0% | dandelion 0,0% | sunflowers 0,0% | roses | ✓ |
| 5 | 25360380_... | **daisy 100,0%** | sunflowers 0,0% | dandelion 0,0% | roses 0,0% | tulips 0,0% | daisy | ✓ |
| 6 | 40410814_... | **sunflowers 99,99%** | dandelion 0,01% | daisy 0,0% | roses 0,0% | tulips 0,0% | sunflowers | ✓ |
| 7 | 40411019_... | **sunflowers 99,45%** | dandelion 0,53% | roses 0,02% | tulips 0,0% | daisy 0,0% | sunflowers | ✓ |
| 8 | 5673551_... | **daisy 99,96%** | sunflowers 0,02% | roses 0,01% | tulips 0,0% | dandelion 0,0% | daisy | ✓ |
| 9 | 8475769_... | **dandelion 97,65%** | sunflowers 2,12% | tulips 0,18% | daisy 0,03% | roses 0,03% | dandelion | ✓ |
|10 | 9818247_... | **dandelion 99,00%** | daisy 0,75% | tulips 0,24% | roses 0,01% | sunflowers 0,0% | dandelion | ✓ |

**Top-1 Accuracy = 10/10 = 100%**
**Top-5 Accuracy = 10/10 = 100%**

Model B (10+10 epok) sklasyfikował wszystkie 10 obrazów poprawnie. Większość predykcji to bardzo
pewne wyniki (>90% dla Top-1). Jedyna niepewność pojawia się przy obrazie nr 3 (tulips/roses ≈ 77%/24%),
co koresponduje z obserwacjami z Confusion Matrix — to najczęstsza para pomyłek modelu.

---
## Zadanie 5 — Odpowiedzi na pytania teoretyczne

### 5a. Dlaczego zamrażamy bazę w Fazie 1?

MobileNetV2 pretrenowany na ImageNet zawiera hierarchię cech wizualnych (krawędzie, tekstury, kształty
i obiekty) wyuczoną na 1,28 mln obrazach z 1 000 klas. Wagi te są bardzo wartościowe i dobrze
uogólniają się na nowe zadania.

Gdybyśmy od razu trenowali całą sieć (z losowo zainicjowaną głową klasyfikacyjną), ogromne gradienty
z głowy zniszczyłyby precyzyjnie wyregulowane wagi bazy — zjawisko zwane **catastrophic forgetting**.

Zamrożenie bazy w Fazie 1 pozwala:**głowie** nauczyć się mapowania cech bazowych na 5 klas kwiatów
bez ryzyka uszkodzenia pretrenowanych reprezentacji. Dopiero gdy głowa jest ustabilizowana,
przechodzimy do fine-tuningu.

### 5b. Co daje fine-tuning w Fazie 2?

Fine-tuning odmraża **ostatnie 30 warstw** bazy MobileNetV2 (warstwy wysokiego poziomu,
uczące cech semantycznych) i trenuje je razem z głową z bardzo małym `lr = 1e-5`.

Efekty:
1. **Dostosowanie wysokopoziomowych cech** do domeny kwiatów — zamiast cech ogólnych z ImageNet,
   sieć uczy się wyodrębniać cechy specyficznie użyteczne dla rozróżniania gatunków kwiatów.
2. **Poprawa metryki** — w naszych eksperymentach fine-tuning podniósł val_acc o ~2–3 pp.
3. **Mały LR** zapobiega zniszczeniu cech wyuczonych w Fazie 1 — zmiany są stopniowe.

W eksperymentach widać wyraźny skok accuracy po przejściu na Fazę 2.

### 5c. Przewaga MobileNetV2 (ImageNet) nad siecią od zera

| Aspekt | MobileNetV2 + ImageNet | Sieć od zera |
|--------|------------------------|---------------|
| Wymagane dane do dobrego wyniku | ~3 670 kwiatów | Dziesiątki–setki tysięcy |
| Czas treningu | Kilka minut (kilka epok) | Godziny / dni |
| Jakość startowych cech | Bogata hierarchia (krawędzie → tekstury → obiekty) | Losowa inicjalizacja |
| Ryzyko przeuczenia | Niskie (cechy odporne na małe datasety) | Wysokie |
| Osiągana dokładność | >90% w 10 epokach | <70% bez dużych danych |

MobileNetV2 wnosi **gotową hierarchię cech wizualnych** — warstwy niskie wykrywają krawędzie
i tekstury, warstwy środkowe — kształty, warstwy wysokie — semantyczne obiekty. Sieć od zera
musiałaby nauczyć się tych wszystkich cech wyłącznie na ~3 670 zdjęciach kwiatów,
co jest praktycznie niemożliwe bez ryzyka przeuczenia.

---
### Wnioski końcowe

1. **Transfer learning z MobileNetV2** okazał się bardzo skuteczny na małym zbiorze flower_photos
   (~3 670 obrazów, 5 klas). Już po 5+5 epokach osiągnęliśmy F1 = 0,922, a po 10+10 — F1 = 0,932.

2. **Dwufazowy trening** jest kluczowy: Faza 1 ustabilizowała głowę klasyfikacyjną, Faza 2
   dostosowała głębokość reprezentacji do domeny i poprawiła wyniki o ~2–3 pp.

3. **Liczba epok ma znaczenie**: 3+3 → niedouczenie (F1=0,905), 5+5 → dobry kompromis (F1=0,922),
   10+10 → najlepszy wynik (F1=0,932) bez przeuczenia.

4. **Confusion Matrix** ujawniła, że model myli głównie pary: roses↔tulips (13+10 błędów)
   i dandelion→daisy (9 błędów). Są to klasy podobne kolorycznie i morfologicznie.

5. **Klasyfikacja 10 obrazów** zakończyła się wynikiem 10/10 Top-1 Accuracy (100%), co potwierdza
   wysoką skuteczność modelu B na danych spoza zbioru treningowego.